# Etapa 2 - Análise exploratória e processamento de dados com Pandas
## SIASUS: produção ambulatorial dos estabelecimentos de Ijuí

## 1. Fonte de dados: Estratégia A (VIEW)

A VIEW deve realiza os relacionamentos com as tabelas descritivas e reduzir a base de dados **apenas para os dados do Município de Ijuí** (`pa_ufmun = '431020'`). 

### Instalação das bibliotecas necessárias para a análise de dados
pip install pandas

## 2. Conexão com o MySQL

Podemos manter a mesma função de conexão utilizada na Aula 11. Executar e informar as credenciais do MySQL quando solicitado.

In [24]:
# conexao.py
from getpass import getpass
from mysql.connector import connect, Error

def usar_conexao(acao):
    try:
        with connect(
            host="localhost",               # Endereço do servidor MySQL (localhost)
            user=input("Usuário: "),        # Solicita ao usuário o nome de login
            password=getpass("Senha: ")     # Solicita a senha de forma oculta
        ) as connection:
            print("Conexão estabelecida:", connection)
            return acao(connection)
    except Error as e:
        print("Ocorreu um erro ao conectar com o BD: ", e)

## 3. Carregar os dados da VIEW para um DataFrame

Todos os exemplos seguintes utilizam DataFrames derivados de `tabela`. DataFrame é uma estrutura de dados bidimensional que organiza as informações em uma tabela de linhas e colunas, semelhante a uma planilha do Excel ou a uma tabela de banco de dados SQL.

**Requisito de infraestrutura:** `fetchall()` carrega os registros na memória antes de construir o DataFrame. 

In [25]:
import pandas as pd

# Consulta SQL única para recuperar todas as colunas e linhas da VIEW.
def carregar_dados(connection):
    with connection.cursor() as cursor:
        cursor.execute("SELECT * FROM dbsia.pa_ijui")
        return pd.DataFrame(cursor.fetchall(), columns=cursor.column_names)

tabela = usar_conexao(carregar_dados)

display(tabela.head(10))

Conexão estabelecida: <mysql.connector.connection_cext.CMySQLConnection object at 0x0000021BED12CE20>


,Ano,Mes,AnoBimestre,AnoTrimestre,Estabelecimento,Procedimento,Ocupacao,pa_sexo,pa_idade,pa_cidpri,Diagnostico,MunicipioEstabelecimento,MunicipioPaciente,pa_qtdpro,pa_qtdapr,pa_valpro,pa_valapr
0,2026,6,2026B3,2026T2,HOSPITAL DE CLINICAS IJUI,QUIMIOTERAPIA DO ADENOCARCINOMA DE PROSTATA RE...,Medico oncologista clinico,M,086,C61,C61 Neopl malig da prostata,Ijui,Santa Barbara,1,1,1062.65,1062.65
1,2026,6,2026B3,2026T2,HOSPITAL DE CLINICAS IJUI,INIBIDOR DA OSTEOLISE,Medico oncologista clinico,M,086,C61,C61 Neopl malig da prostata,Ijui,Santa Barbara,1,1,449.50,449.50
2,2026,6,2026B3,2026T2,HOSPITAL DE CLINICAS IJUI,CONSULTA MEDICA EM ATENCAO ESPECIALIZADA,Medico ginecologista e obstetra,F,043,Z014,Z01.4 Exame ginecologico,Ijui,Humaita,1,1,10.00,10.00
3,2026,6,2026B3,2026T2,HOSPITAL DE CLINICAS IJUI,CONSULTA MEDICA EM ATENCAO ESPECIALIZADA,Medico ginecologista e obstetra,F,042,Z014,Z01.4 Exame ginecologico,Ijui,Humaita,1,0,10.00,0.00
4,2026,6,2026B3,2026T2,HOSPITAL DE CLINICAS IJUI,ULTRASSONOGRAFIA DOPPLER COLORIDO DE VASOS,Medico em cirurgia vascular,M,075,I739,I73.9 Doenc vasculares perifericas NE,Ijui,Santo Antonio,3,3,118.80,118.80
5,2026,6,2026B3,2026T2,HOSPITAL DE CLINICAS IJUI,TOMOGRAFIA COMPUTADORIZADA DO CRANIO,Medico em radiologia e diagnostico por imagem,M,074,0000,CID NAO INFORMADO,Ijui,Independencia,1,1,97.44,97.44
6,2026,6,2026B3,2026T2,HOSPITAL DE CLINICAS IJUI,CONSULTA DE PROFISSIONAIS DE NIVEL SUPERIOR NA...,Enfermeiro,M,054,I516,I51.6 Doenc cardiovascular NE,Ijui,Independencia,1,0,6.30,0.00
7,2026,6,2026B3,2026T2,HOSPITAL DE CLINICAS IJUI,ELETROCARDIOGRAMA,Medico clinico,M,055,0000,CID NAO INFORMADO,Ijui,Santa Maria,1,1,5.15,5.15
8,2026,6,2026B3,2026T2,HOSPITAL DE CLINICAS IJUI,ELETROCARDIOGRAMA,Medico cardiologista,M,054,I516,I51.6 Doenc cardiovascular NE,Ijui,Independencia,1,1,5.15,5.15
9,2026,6,2026B3,2026T2,HOSPITAL DE CLINICAS IJUI,CONSULTA DE PROFISSIONAIS DE NIVEL SUPERIOR NA...,Enfermeiro,M,074,0000,CID NAO INFORMADO,Ijui,Independencia,1,0,6.30,0.00


## 4. Exploração inicial - dimensão

**Perguntas:** quantos registros e variáveis existem? 

In [46]:
print("Quantidade de linhas:", len(tabela))
print("Colunas disponíveis:", list(tabela.columns))
display(tabela.head(5))

Quantidade de linhas: 61900
Colunas disponíveis: ['Ano', 'Mes', 'AnoBimestre', 'AnoTrimestre', 'Estabelecimento', 'Procedimento', 'Ocupacao', 'pa_sexo', 'pa_idade', 'pa_cidpri', 'Diagnostico', 'MunicipioEstabelecimento', 'MunicipioPaciente', 'pa_qtdpro', 'pa_qtdapr', 'pa_valpro', 'pa_valapr', 'IdadeOriginal', 'IdadeNumerica', 'FaixaEtaria']


,Ano,Mes,AnoBimestre,AnoTrimestre,Estabelecimento,Procedimento,Ocupacao,pa_sexo,pa_idade,pa_cidpri,Diagnostico,MunicipioEstabelecimento,MunicipioPaciente,pa_qtdpro,pa_qtdapr,pa_valpro,pa_valapr,IdadeOriginal,IdadeNumerica,FaixaEtaria
0,2026,6,2026B3,2026T2,HOSPITAL DE CLINICAS IJUI,QUIMIOTERAPIA DO ADENOCARCINOMA DE PROSTATA RE...,Medico oncologista clinico,M,086,C61,C61 Neopl malig da prostata,Ijui,Santa Barbara,1.0,1.0,1062.65,1062.65,086,86,60+
1,2026,6,2026B3,2026T2,HOSPITAL DE CLINICAS IJUI,INIBIDOR DA OSTEOLISE,Medico oncologista clinico,M,086,C61,C61 Neopl malig da prostata,Ijui,Santa Barbara,1.0,1.0,449.50,449.50,086,86,60+
2,2026,6,2026B3,2026T2,HOSPITAL DE CLINICAS IJUI,CONSULTA MEDICA EM ATENCAO ESPECIALIZADA,Medico ginecologista e obstetra,F,043,Z014,Z01.4 Exame ginecologico,Ijui,Humaita,1.0,1.0,10.00,10.00,043,43,30–44
3,2026,6,2026B3,2026T2,HOSPITAL DE CLINICAS IJUI,CONSULTA MEDICA EM ATENCAO ESPECIALIZADA,Medico ginecologista e obstetra,F,042,Z014,Z01.4 Exame ginecologico,Ijui,Humaita,1.0,0.0,10.00,0.00,042,42,30–44
4,2026,6,2026B3,2026T2,HOSPITAL DE CLINICAS IJUI,ULTRASSONOGRAFIA DOPPLER COLORIDO DE VASOS,Medico em cirurgia vascular,M,075,I739,I73.9 Doenc vasculares perifericas NE,Ijui,Santo Antonio,3.0,3.0,118.80,118.80,075,75,60+


## 5. Preparação dos tipos de dados

Os campos de quantidade e valor precisam ser numéricos para operações matemáticas. 

In [49]:
colunas_numericas = ["pa_qtdpro", "pa_qtdapr", "pa_valpro", "pa_valapr"]
for coluna in colunas_numericas:
    tabela[coluna] = pd.to_numeric(tabela[coluna])

display(tabela[colunas_numericas].head())

,pa_qtdpro,pa_qtdapr,pa_valpro,pa_valapr
0,1.0,1.0,1062.65,1062.65
1,1.0,1.0,449.50,449.50
2,1.0,1.0,10.00,10.00
3,1.0,0.0,10.00,0.00
4,3.0,3.0,118.80,118.80


## 6. Estatísticas descritivas e distribuições de frequência

`describe()` apresenta contagem não nula, média, desvio-padrão, quartis, mínimo e máximo. A distribuição de frequências de variáveis pode ser obtida com `value_counts()`. 

In [51]:
display(tabela[colunas_numericas].describe())
print("Estabelecimentos mais frequentes por número de registros:")
display(tabela["Estabelecimento"].value_counts().head(10))
print("Distribuição dos códigos de sexo presentes:")
display(tabela["pa_sexo"].value_counts())

,pa_qtdpro,pa_qtdapr,pa_valpro,pa_valapr
count,61900.000000,61900.000000,61900.000000,61900.000000
mean,8.298643,6.397932,98.003098,96.045337
std,55.194760,50.000966,615.559201,615.009292
min,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.630000,0.000000
50%,1.000000,1.000000,6.300000,3.370000
75%,1.000000,1.000000,17.670000,17.670000
max,6031.000000,4871.000000,53758.170000,53758.170000


Estabelecimentos mais frequentes por número de registros:


Estabelecimento
HOSPITAL DE CLINICAS IJUI                           24780
HOSPITAL BOM PASTOR IJUI                             7651
17 CRS IJUI                                          7104
CENTRO DE ATENCAO PSICOSSOCIAL CAPS AD               4004
CER III UNIR                                         3940
APAE IJUI                                            2695
CENTRO DE ATENCAO PSICOSSOCIAL CAPS INFANTIL         2663
POSTO CENTRAL SECRETARIA MUNICIPAL DA SAUDE IJUI     2225
CENTRO AUDITIVO PROAUDI                              2132
CISA                                                 2000
Name: count, dtype: int64

Distribuição dos códigos de sexo presentes:


pa_sexo
F    28821
M    28776
0     4303
Name: count, dtype: int64

## 7. Avaliação da qualidade dos dados

Verificar a quantidade de **NULL / NaN**. 

In [52]:
# Nulos/NaN, por coluna.
nulos = tabela.isna().sum().sort_values(ascending=False)
display(nulos.to_frame("Nulos"))

,Nulos
FaixaEtaria,1072
Procedimento,410
Diagnostico,8
Ano,0
AnoTrimestre,0
AnoBimestre,0
Mes,0
Ocupacao,0
pa_sexo,0
pa_idade,0


## 8. Indicadores gerais da produção ambulatorial

Distinga **número de registros**, **quantidades de procedimentos apresentadas/aprovadas** e **valores apresentados/aprovados**. 

In [54]:
geral = pd.DataFrame([{
    "Registros": len(tabela),
    "QuantidadeApresentada": tabela["pa_qtdpro"].sum(),
    "QuantidadeAprovada": tabela["pa_qtdapr"].sum(),
    "ValorApresentado": tabela["pa_valpro"].sum(),
    "ValorAprovado": tabela["pa_valapr"].sum()
}])
display(geral)

,Registros,QuantidadeApresentada,QuantidadeAprovada,ValorApresentado,ValorAprovado
0,61900,513686.0,396032.0,6066391.74,5945206.35


## 9. Análise temporal - mês

Podemos agrupar **ano + mês**.

In [55]:
mensal = (
    tabela.groupby(["Ano", "Mes"], dropna=False, as_index=False)
    .agg(Registros=("Procedimento", "size"),
         QuantidadeApresentada=("pa_qtdpro", "sum"),
         QuantidadeAprovada=("pa_qtdapr", "sum"),
         ValorApresentado=("pa_valpro", "sum"),
         ValorAprovado=("pa_valapr", "sum"))
    .sort_values(["Ano", "Mes"])
)


print("Produção mensal:")
display(mensal)

Produção mensal:


,Ano,Mes,Registros,QuantidadeApresentada,QuantidadeAprovada,ValorApresentado,ValorAprovado
0,2026,6,61900,513686.0,396032.0,6066391.74,5945206.35


## 10. Produção por estabelecimento

`Estabelecimento` já contém a descrição na VIEW. 
- `dropna=False`: inclui nos agrupamentos os registros que possuem valores ausentes (`NaN`) na coluna utilizada para agrupar. Por padrão, o Pandas exclui esses grupos (`dropna=True`).
- `as_index=False`: mantém a coluna utilizada no agrupamento como uma coluna normal do DataFrame, em vez de transformá-la em índice.

In [57]:
estabelecimentos = (
    tabela.groupby("Estabelecimento", dropna=False, as_index=False)
    .agg(Registros=("Procedimento", "size"),
         QuantidadeApresentada=("pa_qtdpro", "sum"),
         QuantidadeAprovada=("pa_qtdapr", "sum"),
         ValorApresentado=("pa_valpro", "sum"),
         ValorAprovado=("pa_valapr", "sum"))
    .sort_values("QuantidadeAprovada", ascending=False)
)

display(estabelecimentos.head(10))

,Estabelecimento,Registros,QuantidadeApresentada,QuantidadeAprovada,ValorApresentado,ValorAprovado
0,17 CRS IJUI,7104,309205.0,220965.0,1300113.94,1300113.94
13,HOSPITAL DE CLINICAS IJUI,24780,46202.0,38209.0,3284318.54,3189333.12
14,LABORATORIO DE ANALISES CLINICAS UNILAB,309,27552.0,27552.0,135172.34,135172.34
10,CISA,2000,32727.0,26364.0,240329.34,181245.31
17,POSTO CENTRAL SECRETARIA MUNICIPAL DA SAUDE IJUI,2225,30187.0,26077.0,264765.05,231071.84
21,UNIDADE DE PRONTO ATENDIMENTO DR MARTIN CESAR ...,266,22958.0,19917.0,61023.54,58857.24
11,HEMOVITA,195,8082.0,8082.0,40641.50,40641.50
3,CENTRO DE ATENCAO PSICOSSOCIAL CAPS AD,4004,6410.0,4456.0,5131.30,5131.30
9,CER III UNIR,3940,3940.0,3940.0,218608.22,218608.22
5,CENTRO DE ATENCAO PSICOSSOCIAL CAPS INFANTIL,2663,3661.0,3077.0,3323.70,3323.70


## Comandos úteis no Pandas

| Comando | Aplicação |
|---|---|
| `pd.DataFrame()` | Carregar resultado da VIEW |
| `.shape`, `.dtypes`, `.head()`, `.tail()` | Exploração estrutural |
| `.loc[]`, `.iloc[]` | Seleção de linhas e colunas |
| `pd.to_numeric()` | Preparar variáveis numéricas |
| `.describe()`, `.median()`, `.value_counts()` | Estatística descritiva |
| `.isna()`, `.duplicated()`, `.dropna()` | Qualidade e limpeza (sem exclusão automática) |
| `.groupby().agg()` | Agregação por tempo, estabelecimento, procedimento e perfil |
| `.sort_values()`, `.head()` | Rankings |
| `.diff()`, `.pct_change()` | Variação temporal |
| `.str.contains()` | Triagem textual exploratória |
| `pd.cut()` | Construção de faixas etárias |